# Phase 3 — Muon 라인 마무리 (약 1시간, 자투리 시간용)

Muon 을 더 튜닝하는 게 아니다. `docs/PROGRESS.md` §5.2 는 이미 결론이 났다
(전 구성이 SAM 베이스라인 94.69 % 에 7~9 %p 미달).

남기려는 문장 하나 — **"SAM 의 sharpness 감소는 옵티마이저와 무관하게 작동한다"** —
가 노트북 07(Muon v3) 수치에 기대고 있는데, 그 값들은 probe 가 MODE 마다
재생성되던 시절 것이라 **상호 비교가 불가능하다**(§6.3).
노트북 08(v4)은 고정 probe 를 쓰지만 `muon_only` 만 돌렸다.

여기서는 v4 와 같은 조건으로 `muon_only` 와 `sam_always` 를 나란히 돌려
짝이 맞는 비교를 만든다. 우선순위는 낮다 — Phase 1 이 GPU 를 놀리는 틈에 끼우면 된다.

In [ ]:
# ══ 1. 셋업 ═══════════════════════════════════════════════════════════════
import os, sys, math, json, time, glob, random, tarfile, urllib.request
from datetime import datetime
from collections import defaultdict

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

DATA_DIR = os.environ.get('DATA_DIR', '/content/data')
OUT_DIR  = os.environ.get('OUT_DIR',  '/content/out')
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device,
      f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else '')
print('torch  :', torch.__version__)
print('출력   :', OUT_DIR)

In [ ]:
# ══ 2. 재현성 ═════════════════════════════════════════════════════════════
# 노트북판의 가장 큰 결함이 "Stage 2 에 시드가 없다"는 것이었다.
# sam_off 셀이 RNG 를 소모한 뒤 sam_on 셀이 시작돼, 두 arm 이 서로 다른
# 데이터 순서를 봤다. 아래 세 함수로 그 경로를 원천 차단한다.
def seed_everything(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

def worker_init(worker_id):
    random.seed(torch.initial_seed() % 2**32 + worker_id)

print('시드 유틸 준비 완료')

In [ ]:
# ══ 3. 데이터 ═════════════════════════════════════════════════════════════
CIFAR_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
C10_MEAN, C10_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)

def prepare_data(data_dir=DATA_DIR):
    root = os.path.join(data_dir, 'cifar10')
    if not os.path.exists(root):
        tgz = os.path.join(data_dir, 'cifar10.tgz')
        if not os.path.exists(tgz):
            print('CIFAR-10 다운로드 중...', flush=True)
            urllib.request.urlretrieve(CIFAR_URL, tgz)
        print('압축 해제 중...', flush=True)
        with tarfile.open(tgz, 'r:gz') as tar:
            tar.extractall(path=data_dir, filter='data')
    return root

def build_loaders(seed, batch_size=128, num_workers=2):
    root = prepare_data()
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    tf_eval = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(C10_MEAN, C10_STD)])
    train = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_train)
    test  = datasets.ImageFolder(os.path.join(root, 'test'),  transform=tf_eval)
    # probe 전용: 같은 train 이미지지만 augmentation 없이 결정론적으로 읽는다
    probe_src = datasets.ImageFolder(os.path.join(root, 'train'), transform=tf_eval)
    tl = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=True, generator=loader_generator(seed),
                    worker_init_fn=worker_init)
    vl = DataLoader(test, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tl, vl, probe_src

def make_probe(probe_src, n=384, batch_size=128, probe_seed=1234):
    """sharpness 측정용 고정 배치.
    인덱스는 별도 Generator 로 뽑고 이미지는 augmentation 이 없으므로,
    세션이 달라도 항상 같은 텐서가 나온다."""
    g = torch.Generator().manual_seed(probe_seed)
    idx = torch.randperm(len(probe_src), generator=g)[:n].tolist()
    loader = DataLoader(Subset(probe_src, idx), batch_size=batch_size, shuffle=False)
    return [(x.to(device), y.to(device)) for x, y in loader]

print('데이터 함수 준비 완료')

In [ ]:
# ══ 4. 옵티마이저 + SAM ═══════════════════════════════════════════════════
class Lion(optim.Optimizer):
    """update = sign(b1*m + (1-b1)*g),  m = b2*m + (1-b2)*g"""
    def __init__(self, params, lr=2e-4, betas=(0.9, 0.99), weight_decay=0.5):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            b1, b2 = group['betas']
            for p in group['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'exp_avg' not in st: st['exp_avg'] = torch.zeros_like(p)
                m, g = st['exp_avg'], p.grad
                if wd: p.mul_(1 - lr * wd)
                p.add_(m.mul(b1).add_(g, alpha=1-b1).sign_(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1-b2)


class SAMAscent:
    """SAM 을 '옵티마이저'가 아니라 '그래디언트 변환기'로 분리한 것.
    perturbation 만 담당하고 step 은 외부 옵티마이저가 하므로,
    옵티마이저를 교체하지 않고 SAM 을 켜고 끌 수 있다 → 모멘텀이 연속된다."""
    def __init__(self, params, rho=0.0):
        self.params = [p for p in params if p.requires_grad]
        self.rho = rho; self._e_w = {}

    def set_rho(self, rho): self.rho = rho

    @torch.no_grad()
    def grad_norm(self):
        gs = [p.grad.norm(p=2) for p in self.params if p.grad is not None]
        return torch.norm(torch.stack(gs), p=2) if gs else torch.tensor(0.0)

    @torch.no_grad()
    def ascent(self):
        if self.rho <= 0: return False
        gn = self.grad_norm()
        if gn.item() < 1e-12: return False
        scale = self.rho / (gn + 1e-12)
        self._e_w.clear()
        for p in self.params:
            if p.grad is None: continue
            e = p.grad * scale.to(p.device); p.add_(e); self._e_w[p] = e
        return True

    @torch.no_grad()
    def restore(self):
        for p, e in self._e_w.items(): p.sub_(e)
        self._e_w.clear()

def build_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)

print('Lion / SAMAscent / ResNet-18 준비 완료')

In [ ]:
# ══ 5. WSD 스케줄 + sharpness 측정 ════════════════════════════════════════
def wsd_lr(epoch, total_epochs, peak_lr, warmup_epochs, decay_epochs):
    """warmup(선형 상승) → stable(유지) → decay(선형 하강)"""
    stable_end = total_epochs - decay_epochs
    if epoch < warmup_epochs:
        return peak_lr * (epoch + 1) / warmup_epochs
    if epoch < stable_end:
        return peak_lr
    prog = min(1.0, max(0.0, (epoch - stable_end + 1) / decay_epochs))
    return peak_lr * (1.0 - prog)


@torch.no_grad()
def _perturb(e_ws, sign):
    for p, e in e_ws: p.add_(e, alpha=sign)


def measure_sharpness(model, criterion, probe, rho=0.05):
    """세 값을 함께 반환한다.

    one_sided : L(w+e) − L(w)
        테일러 전개하면 ≈ rho*||g|| + (1/2)rho^2 * (g'Hg)/||g||^2 이다.
        즉 1차항이 그래디언트 노름이라, 이 값은 곡률이 아니라 ||g|| 를
        상당 부분 재고 있다. 과거 결과와 비교하려고 유지한다.

    symmetric : (1/2)(L(w+e) + L(w−e)) − L(w)
        1차항이 상쇄되어 곡률 항만 남는다. 이쪽이 우리가 재려던 것.

    grad_norm : 위 해석을 사후 검증하기 위한 원자료 (Gate A).
    """
    was_training = model.training
    model.eval()
    one, sym, gns = [], [], []
    for x, y in probe:
        with torch.no_grad():
            l0 = criterion(model(x), y).item()
        model.zero_grad(set_to_none=True)
        criterion(model(x), y).backward()
        gs = [p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]
        gn = torch.norm(torch.stack(gs), p=2); gns.append(gn.item())
        scale = rho / (gn + 1e-12)
        e_ws = [(p, p.grad * scale) for p in model.parameters() if p.grad is not None]
        with torch.no_grad():
            _perturb(e_ws, +1.0);  lp = criterion(model(x), y).item()
            _perturb(e_ws, -2.0);  lm = criterion(model(x), y).item()
            _perturb(e_ws, +1.0)                     # 원위치 복원
        model.zero_grad(set_to_none=True)
        one.append(lp - l0); sym.append(0.5 * (lp + lm) - l0)
    if was_training: model.train()
    n = len(one)
    return sum(one)/n, sum(sym)/n, sum(gns)/n


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        total += y.size(0); correct += model(x).max(1)[1].eq(y).sum().item()
    return 100.0 * correct / total

print('스케줄 / sharpness / 평가 함수 준비 완료')

In [ ]:
# ══ 6. 실시간 그래프 ══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from IPython.display import clear_output

LIVE = {'on': True, 'every': 1}       # every=5 로 두면 5에폭마다 갱신

def live_plot(tag, history, sharp):
    if not LIVE['on'] or len(history['epoch']) % LIVE['every']: return
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history['epoch'], history['train_acc'], lw=1.2, label='train')
    ax[0].plot(history['epoch'], history['test_acc'],  lw=1.6, label='test')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
    ax[0].set_title(f'{tag} — test {history["test_acc"][-1]:.2f}%')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if sharp['epoch']:
        ax[1].plot(sharp['epoch'], sharp['one_sided'], 'o-', ms=3,
                   color='tab:orange', label='one-sided')
        ax[1].plot(sharp['epoch'], sharp['symmetric'], 's-', ms=3,
                   color='tab:purple', label='symmetric (curvature)')
        ax[1].set_xlabel('epoch'); ax[1].set_ylabel('sharpness')
        ax[1].set_title('sharpness (lower = flatter)')
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(history['epoch'], history['grad_norm'], color='tab:brown', lw=1.2)
    ax[2].set_xlabel('epoch'); ax[2].set_ylabel('grad norm', color='tab:brown')
    ax[2].grid(alpha=.3); ax[2].set_title('grad norm & rho')
    a2 = ax[2].twinx(); a2.plot(history['epoch'], history['rho'],
                                color='tab:green', lw=1.4)
    a2.set_ylabel('rho', color='tab:green')
    plt.tight_layout(); plt.show()
    print(f'[{tag}] epoch {history["epoch"][-1]} | '
          f'loss {history["train_loss"][-1]:.4f} | '
          f'test {history["test_acc"][-1]:.2f}% | '
          f'{sum(history["time"])/60:.1f}분 경과', flush=True)

print('실시간 그래프 준비 완료 (끄려면 LIVE["on"] = False)')

In [ ]:
# ══ Muon 옵티마이저 ═══════════════════════════════════════════════════════
@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Newton-Schulz 반복으로 gradient 를 직교화한다 (Muon 의 핵심 연산)."""
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16(); X /= (X.norm() + eps)
    t = G.size(0) > G.size(1)
    if t: X = X.T
    for _ in range(steps):
        A = X @ X.T; B = b*A + c*A@A; X = a*X + B@X
    if t: X = X.T
    return X.to(G.dtype)


class Muon(optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      ns_steps=ns_steps, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                g = p.grad
                if g is None: continue
                st = self.state[p]
                if 'momentum_buffer' not in st: st['momentum_buffer'] = torch.zeros_like(g)
                buf = st['momentum_buffer']
                shape = g.shape
                if g.ndim > 2: g = g.view(g.size(0), -1)
                go = zeropower_via_newtonschulz5(g, steps=group['ns_steps']).view(shape)
                buf.mul_(group['momentum']).add_(go)
                upd = go + group['momentum']*buf if group['nesterov'] else buf
                if group['weight_decay']: p.mul_(1 - group['lr']*group['weight_decay'])
                p.add_(upd, alpha=-group['lr'])


def split_params(model, scope='hidden_only', verbose=True):
    """'hidden_only' = Muon 공식 가이드. 분류기 헤드(fc)와 stem conv(conv1)를 제외.
    노트북 08(v4)과 같은 조건이다."""
    muon_p, adamw_p, excl = [], [], []
    for name, p in model.named_parameters():
        if p.ndim < 2: adamw_p.append(p); continue
        if scope == 'hidden_only' and (name.startswith('fc.') or name.startswith('conv1.')):
            adamw_p.append(p); excl.append((name, tuple(p.shape))); continue
        muon_p.append(p)
    if verbose:
        print(f"scope='{scope}'  Muon {len(muon_p)}텐서/"
              f"{sum(p.numel() for p in muon_p):,}  AdamW {len(adamw_p)}텐서/"
              f"{sum(p.numel() for p in adamw_p):,}")
        for n, s in excl: print(f'   제외: {n} {s}')
    return muon_p, adamw_p

print('Muon 준비 완료')

In [ ]:
# ══ 실행 함수 ═════════════════════════════════════════════════════════════
def run_muon(mode, seed=42, epochs=50, scope='hidden_only', rho_max=0.05,
             ramp_epochs=5, muon_lr=0.02, muon_wd=0.0, adamw_lr=3e-4,
             grad_clip=1.0, sharpness_interval=2, sharpness_rho=0.05,
             batch_size=128):
    """mode: 'muon_only' | 'sam_always'"""
    out = os.path.join(OUT_DIR, f'muon_{mode}_{scope}_seed{seed}.json')
    if os.path.exists(out):
        print(f'skip {mode} (이미 있음)'); return json.load(open(out, encoding='utf-8'))

    seed_everything(seed)
    tl, vl, ps = build_loaders(seed, batch_size)
    probe = make_probe(ps)
    model = build_model()
    mp, ap = split_params(model, scope)
    om = Muon(mp, lr=muon_lr, momentum=0.95, weight_decay=muon_wd)
    oa = optim.AdamW(ap, lr=adamw_lr, weight_decay=0.01)
    sm = optim.lr_scheduler.CosineAnnealingLR(om, T_max=epochs)
    sa = optim.lr_scheduler.CosineAnnealingLR(oa, T_max=epochs)
    sam = SAMAscent(model.parameters()); crit = nn.CrossEntropyLoss()

    hist = {k: [] for k in ['epoch','phase','lr','rho','train_loss','train_acc',
                            'test_acc','grad_norm','ascent_gap','time']}
    sharp = {'epoch': [], 'one_sided': [], 'symmetric': [], 'probe_grad_norm': []}

    print(f'\n=== Muon [{mode} / {scope} / seed {seed}] {epochs} epochs ===\n', flush=True)
    for epoch in range(epochs):
        rho_now = (rho_max * min(1.0, (epoch+1)/max(1, ramp_epochs))
                   if mode == 'sam_always' else 0.0)
        sam.set_rho(rho_now)
        model.train()
        rl = rg = rn = 0.0; correct = total = nb = 0; t0 = time.time()
        for x, y in tl:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            om.zero_grad(set_to_none=True); oa.zero_grad(set_to_none=True)
            o = model(x); loss = crit(o, y); loss.backward()
            total += y.size(0); correct += o.max(1)[1].eq(y).sum().item()
            rn += sam.grad_norm().item()
            gap = 0.0
            if sam.ascent():
                om.zero_grad(set_to_none=True); oa.zero_grad(set_to_none=True)
                la = crit(model(x), y); la.backward(); gap = la.item()-loss.item()
                sam.restore()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            om.step(); oa.step(); rl += loss.item(); rg += gap; nb += 1
        sm.step(); sa.step()
        dt = time.time()-t0
        te = evaluate(model, vl)
        for k, v in [('epoch',epoch+1),('phase',mode),('lr',om.param_groups[0]['lr']),
                     ('rho',rho_now),('train_loss',rl/nb),('train_acc',100.0*correct/total),
                     ('test_acc',te),('grad_norm',rn/nb),('ascent_gap',rg/nb),('time',dt)]:
            hist[k].append(v)
        extra = ''
        if (epoch+1) % sharpness_interval == 0 or epoch == epochs-1:
            o_, s_, g_ = measure_sharpness(model, crit, probe, sharpness_rho)
            sharp['epoch'].append(epoch+1); sharp['one_sided'].append(o_)
            sharp['symmetric'].append(s_); sharp['probe_grad_norm'].append(g_)
            extra = f' | Sharp1 {o_:.4f} | SharpSym {s_:.6f}'
        print(f'[{mode}] Ep [{epoch+1}/{epochs}] | {dt:.1f}s | Loss {rl/nb:.4f} | '
              f'Train {100.0*correct/total:.2f}% | Test {te:.2f}% | '
              f'rho {rho_now:.4f}{extra}', flush=True)
        try: live_plot(mode, hist, sharp)
        except Exception as e: print('  (그래프 오류 무시:', e, ')', flush=True)

    best = max(hist['test_acc'])
    rec = {'mode': mode, 'scope': scope, 'seed': seed, 'epochs': epochs,
           'rho_max': rho_max if mode == 'sam_always' else 0.0,
           'results': {'final_train_acc': hist['train_acc'][-1],
                       'final_test_acc': hist['test_acc'][-1], 'best_test_acc': best,
                       'final_gap': hist['train_acc'][-1]-hist['test_acc'][-1],
                       'final_one_sided': sharp['one_sided'][-1],
                       'final_symmetric': sharp['symmetric'][-1],
                       'final_probe_grad_norm': sharp['probe_grad_norm'][-1],
                       'total_time_min': sum(hist['time'])/60},
           'history': hist, 'sharpness_history': sharp,
           'timestamp': datetime.now().isoformat(timespec='seconds')}
    json.dump(rec, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    print(f'\n저장: {out}', flush=True)
    return rec

print('run_muon 준비 완료')

In [ ]:
# ══ 실행 (두 MODE 연속) ═══════════════════════════════════════════════════
EPOCHS = 50          # 스모크로 먼저 보려면 2 로 낮출 것
SEED   = 42

recs = {m: run_muon(m, seed=SEED, epochs=EPOCHS) for m in ['muon_only', 'sam_always']}

In [ ]:
# ══ 비교 — 이 표 한 장이 §5.2 의 한 문단을 대체한다 ═══════════════════════
V4_REF = {'final_test': 85.29, 'one_sided': 0.3738}   # 노트북 08 실측
print(f'{"mode":<14}{"Test":>9}{"Best":>9}{"Sharp1":>10}{"SharpSym":>12}{"|g|":>9}{"min":>7}')
print('-'*70)
for m, rec in recs.items():
    r = rec['results']
    print(f'{m:<14}{r["final_test_acc"]:>9.2f}{r["best_test_acc"]:>9.2f}'
          f'{r["final_one_sided"]:>10.4f}{r["final_symmetric"]:>12.6f}'
          f'{r["final_probe_grad_norm"]:>9.3f}{r["total_time_min"]:>7.1f}')
print('-'*70)
print(f'참고) 노트북 08(v4) muon_only: Test {V4_REF["final_test"]:.2f}% / '
      f'Sharp1 {V4_REF["one_sided"]:.4f}')

a, b = recs['muon_only']['results'], recs['sam_always']['results']
d_sym = b['final_symmetric'] - a['final_symmetric']
print(f'\nSAM 효과: SharpSym {d_sym:+.6f} | '
      f'Test {b["final_test_acc"]-a["final_test_acc"]:+.2f}%p')
print('→ ' + ('Muon 에서도 SAM 이 곡률을 낮춘다. "옵티마이저 무관"을 측정값으로 쓸 수 있다.'
               if d_sym < 0 else
               'Muon 에서는 SAM 이 곡률을 낮추지 못했다. 해당 문장을 Lion 한정으로 축소할 것.'))

In [ ]:
# ══ ★ 회수 (세션 끄기 전 반드시 실행) ═════════════════════════════════════
# Colab VM 은 임시다. Kaggle 에서 이미 한 번 전부 날렸다 (PROGRESS.md §8).
arc = os.path.join(OUT_DIR, 'phase3_muon_results.tar.gz')
with tarfile.open(arc, 'w:gz') as t:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.json'))
                    + glob.glob(os.path.join(OUT_DIR, '*.png'))):
        t.add(f, arcname=os.path.basename(f))
print(f'{arc}  ({os.path.getsize(arc)/1024:.0f} KB)')
try:
    from google.colab import files; files.download(arc)
except Exception as e:
    print('브라우저 다운로드 불가:', e)
    print('CLI 로:  colab download -s <세션>', arc, './results/')